# Benchmark AMT completo — Objetivo específico 1 / Hito H2

Evaluación comparativa de arquitecturas de Transcripción Musical Automática sobre
obras completas de la partición de prueba de **MAESTRO v3.0.0**.

* ≥ 20 obras completas de la partición de prueba.
* Contraste de al menos tres familias arquitectónicas del estado del arte.
* Umbrales para la arquitectura elegida: **F1_onset ≥ 80 %**, **F1_note (con offset) ≥ 75 %**, **NER ≤ 25 %**.
* Métricas calculadas con `mir_eval` (Raffel et al., 2014).

## Modelos evaluados

| Modelo | Familia arquitectónica | Estado |
|---|---|---|
| Kong et al. (2021) — `piano_transcription_inference` | CRNN doble objetivo, variante de alta resolución | Empírico |
| Basic Pitch (Bittner et al., 2022) | CNN ligera multi-tarea, instrumento-agnóstica | Empírico |
| hFT-Transformer (Toyama et al., 2023) | Transformer jerárquico frecuencia-tiempo | Empírico (checkpoint oficial ISMIR 2023) |
| Onsets and Frames (Hawthorne et al., 2018) | CRNN doble objetivo (original) | Línea base bibliográfica |

Onsets and Frames no se evalúa empíricamente porque su implementación original depende
de TensorFlow 1.15, incompatible con el entorno de Kaggle (Python 3.12 / CUDA 12); se
incluye como línea base bibliográfica con sus métricas publicadas, tal como se documenta
en `docs/literature_review_amt.md`.

## Entorno

* Kaggle Notebooks con el dataset `alonhaviv/the-maestro-dataset-v3-0-0` montado como *input*.
* Acelerador GPU T4 e internet habilitado (descarga de checkpoints).
* Los resultados por obra se guardan en `/kaggle/working/benchmark_full/` como JSON:
  el notebook es **reanudable** — si la sesión muere, volver a ejecutar y se retoman solo
  las obras pendientes.


## 1. Instalación de dependencias

Se instala con `--no-deps` donde es necesario para evitar que pip degrade numpy a una versión incompatible con el Python del entorno de Kaggle (misma estrategia documentada en `docs/literature_review_amt.md` §6.3).

In [4]:
import subprocess, sys

def pip(*args):
    print(" ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("mir_eval")
pip("--no-deps", "piano_transcription_inference", "torchlibrosa")
pip("--no-deps", "basic-pitch", "tensorflow-io-gcs-filesystem")
pip("--no-deps", "pretty_midi", "resampy")
pip("librosa", "soundfile")


mir_eval
--no-deps piano_transcription_inference torchlibrosa
--no-deps basic-pitch tensorflow-io-gcs-filesystem
--no-deps pretty_midi resampy
librosa soundfile


## 2. Configuración

In [5]:
import json, os, random, time, glob
from pathlib import Path

N_WORKS = 20            # minimo exigido por el Objetivo especifico 1
SEED = 22779            # carne del estudiante: seleccion determinista
ONSET_TOL = 0.05        # ±50 ms (convencion mir_eval / Marco Metodologico)
OFFSET_RATIO = 0.2      # offset: max(50 ms, 20% de la duracion)

# Umbrales del Objetivo especifico 1 (hito H2)
TH_F1_ONSET = 0.80
TH_F1_NOTE = 0.75
TH_NER = 0.25

DATASET_ROOT = None
for cand in glob.glob("/kaggle/input/*/"):
    if glob.glob(cand + "**/maestro-v3.0.0.csv", recursive=True):
        DATASET_ROOT = Path(glob.glob(cand + "**/maestro-v3.0.0.csv", recursive=True)[0]).parent
        break
assert DATASET_ROOT is not None, "Montar el dataset the-maestro-dataset-v3-0-0 como input"
print("Dataset:", DATASET_ROOT)

RESULTS_DIR = Path("/kaggle/working/benchmark_full")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


Dataset: /kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0


## 3. Selección de obras

Se toman `N_WORKS` obras completas de la partición `test`, elegidas con muestreo aleatorio determinista (semilla fija) para no sesgar por compositor ni duración. La lista queda registrada en los resultados.

In [6]:
import csv

with open(DATASET_ROOT / "maestro-v3.0.0.csv", newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r["split"] == "test"]

print(f"Obras en la particion de prueba: {len(rows)}")
rng = random.Random(SEED)
works = rng.sample(rows, N_WORKS)

def resolve(rel):
    p = DATASET_ROOT / rel
    if p.exists():
        return p
    hits = glob.glob(str(DATASET_ROOT / "**" / Path(rel).name), recursive=True)
    assert hits, f"No se encontro {rel}"
    return Path(hits[0])

total_min = 0.0
for i, w in enumerate(works):
    total_min += float(w["duration"]) / 60
    print(f"{i+1:2d}. [{float(w['duration'])/60:5.1f} min] "
          f"{w['canonical_composer']} — {w['canonical_title'][:60]}")
print(f"\nDuracion total: {total_min:.1f} min")


Obras en la particion de prueba: 177
 1. [  5.0 min] Joseph Haydn — Sonata in C Minor, First Movement
 2. [  3.9 min] Franz Schubert — "Gretchen am Spinnrade"
 3. [  4.9 min] Sergei Rachmaninoff — Etude Tableau Op. 39 No. 5
 4. [  3.8 min] Frédéric Chopin — Nocturne in F-sharp Major, Op. 15, No. 2
 5. [  4.0 min] Frédéric Chopin — Nocturne Op. 15 No. 1 in F Major
 6. [ 10.2 min] Frédéric Chopin — Rondo in C Minor, Op. 1
 7. [  1.6 min] Domenico Scarlatti — Sonata in D Minor, K. 9 L. 413
 8. [  4.6 min] Ludwig van Beethoven — Sonata No. 16 in G Major, Op. 31 No. 1, First movement
 9. [  4.9 min] Robert Schumann — Toccata in C Major, Op. 7
10. [  3.7 min] Franz Liszt — Au bord d'une source
11. [  4.7 min] Ludwig van Beethoven — Sonata No. 16 in G Major, Op. 31 No. 1, First Movement
12. [ 14.2 min] Claude Debussy — Estampes (Complete)
13. [ 10.5 min] Franz Liszt — Hungarian Rhapsody No. 9
14. [ 20.1 min] Ludwig van Beethoven — Sonata No. 18 in E-flat Major, Op. 31, No. 3 (Complete)
15. [ 

## 4. Métricas

* **F1_onset**: F1 a nivel de *onset* (±50 ms) con altura correcta, `offset_ratio=None`.
* **F1_note**: F1 a nivel de nota con *offset* (±50 ms de onset; offset dentro de
  max(50 ms, 20 % de la duración)).
* **NER (Note Error Rate)**: con el emparejamiento onset+altura de `mir_eval`,
  `NER = (FN + FP) / N_ref`, donde FN son notas de referencia sin pareja y FP notas
  estimadas sin pareja. Una sustitución (altura equivocada) cuenta como un FN más un
  FP. Es la definición complementaria del F1 de onset documentada en el Marco
  Metodológico.
* **Latencia normalizada**: tiempo de inferencia dividido entre la duración del audio
  (criterio del hito H4: ≤ 1.5).

In [7]:
import numpy as np
import pretty_midi
import mir_eval.transcription as mt

def load_reference(midi_path):
    pm = pretty_midi.PrettyMIDI(str(midi_path))
    notes = [n for inst in pm.instruments if not inst.is_drum for n in inst.notes]
    notes.sort(key=lambda n: (n.start, n.pitch))
    intervals = np.array([[n.start, n.end] for n in notes])
    pitches = np.array([pretty_midi.note_number_to_hz(n.pitch) for n in notes])
    return intervals, pitches

def evaluate(ref_i, ref_p, est_i, est_p):
    if len(est_i) == 0:
        return {"f1_onset": 0.0, "p_onset": 0.0, "r_onset": 0.0,
                "f1_note": 0.0, "ner": 1.0, "n_ref": len(ref_i), "n_est": 0}
    p_on, r_on, f_on, _ = mt.precision_recall_f1_overlap(
        ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL, offset_ratio=None)
    _, _, f_note, _ = mt.precision_recall_f1_overlap(
        ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL,
        offset_ratio=OFFSET_RATIO, offset_min_tolerance=0.05)
    matching = mt.match_notes(ref_i, ref_p, est_i, est_p,
                              onset_tolerance=ONSET_TOL, offset_ratio=None)
    tp = len(matching)
    fn = len(ref_i) - tp
    fp = len(est_i) - tp
    ner = (fn + fp) / len(ref_i) if len(ref_i) else 0.0
    return {"f1_onset": f_on, "p_onset": p_on, "r_onset": r_on,
            "f1_note": f_note, "ner": ner, "n_ref": len(ref_i), "n_est": len(est_i)}


ModuleNotFoundError: No module named 'mido'

## 5. Runners de los modelos

Cada *runner* expone `transcribe(audio_path) -> (intervals, pitches_hz)`. El modelo se
carga una sola vez (la carga no cuenta en la latencia por obra).

In [ ]:
class KongRunner:
    """Kong et al. (2021) — CRNN de alta resolucion (PyTorch, Apache 2.0)."""
    name = "Kong et al. (2021)"

    def __init__(self):
        import torch
        from piano_transcription_inference import PianoTranscription, sample_rate
        self.sample_rate = sample_rate
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = PianoTranscription(device=device)

    def transcribe(self, audio_path):
        import librosa
        audio, _ = librosa.load(str(audio_path), sr=self.sample_rate, mono=True)
        out = self.model.transcribe(audio, "/tmp/_kong_tmp.mid")
        events = out["est_note_events"]
        intervals = np.array([[e["onset_time"], e["offset_time"]] for e in events])
        pitches = np.array([pretty_midi.note_number_to_hz(e["midi_note"]) for e in events])
        return intervals, pitches


In [ ]:
class BasicPitchRunner:
    """Basic Pitch (Bittner et al., 2022) — CNN ligera multi-tarea (TF2, Apache 2.0)."""
    name = "Basic Pitch (2022)"

    def __init__(self):
        from basic_pitch.inference import predict
        from basic_pitch import ICASSP_2022_MODEL_PATH
        self._predict = predict
        self._model_path = ICASSP_2022_MODEL_PATH

    def transcribe(self, audio_path):
        _, midi_data, _ = self._predict(str(audio_path), self._model_path)
        notes = [n for inst in midi_data.instruments for n in inst.notes]
        intervals = np.array([[n.start, n.end] for n in notes])
        pitches = np.array([pretty_midi.note_number_to_hz(n.pitch) for n in notes])
        return intervals, pitches


In [ ]:
class HFTRunner:
    """hFT-Transformer (Toyama et al., 2023) — Transformer jerarquico (PyTorch, MIT).

    Usa el checkpoint oficial del release ISMIR 2023 (MAESTRO-V3). Si el entorno no
    permite ejecutarlo, el benchmark continua con los otros dos modelos y se deja
    constancia del fallo.
    """
    name = "hFT-Transformer (2023)"
    REPO = "https://github.com/sony/hFT-Transformer.git"
    CKPT = "https://github.com/sony/hFT-Transformer/releases/download/ismir2023/checkpoint.zip"

    def __init__(self):
        import subprocess, sys
        work = Path("/kaggle/working/hFT-Transformer")
        if not work.exists():
            subprocess.run(["git", "clone", "--depth", "1", self.REPO, str(work)], check=True)
        ckpt_dir = work / "checkpoint"
        if not (ckpt_dir / "MAESTRO-V3").exists():
            subprocess.run(["wget", "-q", self.CKPT, "-O", "/tmp/ckpt.zip"], check=True)
            subprocess.run(["unzip", "-oq", "/tmp/ckpt.zip", "-d", str(ckpt_dir)], check=True)
        model_file = next((ckpt_dir / "MAESTRO-V3").glob("*.pkl"))
        sys.path.insert(0, str(work))
        os.chdir(work)   # `from model import amt` y rutas relativas del repo
        with open(work / "corpus" / "config.json", encoding="utf-8") as f:
            config = json.load(f)
        from model import amt
        self.amt = amt.AMT(config, str(model_file), verbose_flag=False)

    def transcribe(self, audio_path):
        feature = self.amt.wav2feature(str(audio_path))
        out = self.amt.transcript(feature, mode="combination")
        onset, offset, mpe, velocity = out[4], out[5], out[6], out[7]   # salida 2nd
        notes = self.amt.mpe2note(a_onset=onset, a_offset=offset, a_mpe=mpe,
                                  a_velocity=velocity,
                                  thred_onset=0.5, thred_offset=0.5, thred_mpe=0.5)
        intervals = np.array([[n["onset"], n["offset"]] for n in notes])
        pitches = np.array([pretty_midi.note_number_to_hz(n["pitch"]) for n in notes])
        return intervals, pitches


## 6. Ejecución del benchmark

Un JSON por (modelo, obra) en `RESULTS_DIR`; las combinaciones ya calculadas se omiten al re-ejecutar.

In [ ]:
RUNNERS = [KongRunner, BasicPitchRunner, HFTRunner]

def run_benchmark():
    cwd = os.getcwd()
    for runner_cls in RUNNERS:
        os.chdir(cwd)
        pending = [w for w in works
                   if not (RESULTS_DIR / f"{runner_cls.__name__}__{Path(w['midi_filename']).stem}.json").exists()]
        if not pending:
            print(f"[{runner_cls.name}] completo (cache)")
            continue
        try:
            runner = runner_cls()
        except Exception as exc:
            print(f"[{runner_cls.name}] NO EVALUADO: {type(exc).__name__}: {exc}")
            continue
        for w in pending:
            stem = Path(w["midi_filename"]).stem
            out_file = RESULTS_DIR / f"{runner_cls.__name__}__{stem}.json"
            audio = resolve(w["audio_filename"])
            midi = resolve(w["midi_filename"])
            try:
                t0 = time.time()
                est_i, est_p = runner.transcribe(audio)
                latency = time.time() - t0
                ref_i, ref_p = load_reference(midi)
                m = evaluate(ref_i, ref_p, est_i, est_p)
                m.update({"model": runner.name, "work": stem,
                          "composer": w["canonical_composer"],
                          "title": w["canonical_title"],
                          "duration_s": float(w["duration"]),
                          "latency_s": latency,
                          "latency_norm": latency / float(w["duration"])})
                out_file.write_text(json.dumps(m, indent=2))
                print(f"[{runner.name}] {stem[:40]:40s} "
                      f"F1on={m['f1_onset']:.3f} F1note={m['f1_note']:.3f} "
                      f"NER={m['ner']:.3f} lat={m['latency_norm']:.2f}x")
            except Exception as exc:
                print(f"[{runner.name}] {stem}: ERROR {type(exc).__name__}: {exc}")
    os.chdir(cwd)

run_benchmark()


## 7. Agregación y verificación de umbrales

In [ ]:
import pandas as pd

records = [json.loads(p.read_text()) for p in RESULTS_DIR.glob("*.json")]
df = pd.DataFrame(records)
assert not df.empty, "Sin resultados: revisar la celda anterior"

agg = df.groupby("model").agg(
    obras=("work", "count"),
    f1_onset=("f1_onset", "mean"), f1_onset_std=("f1_onset", "std"),
    f1_note=("f1_note", "mean"), f1_note_std=("f1_note", "std"),
    ner=("ner", "mean"), ner_std=("ner", "std"),
    lat_norm=("latency_norm", "mean"), lat_norm_p90=("latency_norm", lambda s: s.quantile(0.9)),
).round(4)
display(agg)

print("\nVerificacion de umbrales del Objetivo especifico 1 (hito H2):")
for model, r in agg.iterrows():
    ok = (r.f1_onset >= TH_F1_ONSET and r.f1_note >= TH_F1_NOTE and r.ner <= TH_NER)
    print(f"  {'CUMPLE  ' if ok else 'NO cumple'} {model}: "
          f"F1_onset={r.f1_onset:.2%} (≥{TH_F1_ONSET:.0%}), "
          f"F1_note={r.f1_note:.2%} (≥{TH_F1_NOTE:.0%}), "
          f"NER={r.ner:.2%} (≤{TH_NER:.0%})")

df.to_csv("/kaggle/working/benchmark_full_por_obra.csv", index=False)
agg.to_csv("/kaggle/working/benchmark_full_agregado.csv")


## 8. Tabla final (incluye línea base bibliográfica)

Salida en Markdown para incorporarla a `docs/literature_review_amt.md` §7.

In [ ]:
lines = [
    "| Modelo | Obras | F1_onset | F1_note (con offset) | NER | Latencia norm. (media / p90) |",
    "|---|:---:|:---:|:---:|:---:|:---:|",
]
for model, r in agg.iterrows():
    lines.append(
        f"| {model} | {int(r.obras)} | {r.f1_onset:.2%} ± {r.f1_onset_std:.2%} "
        f"| {r.f1_note:.2%} ± {r.f1_note_std:.2%} | {r.ner:.2%} "
        f"| {r.lat_norm:.3f} / {r.lat_norm_p90:.3f} |")
lines.append(
    "| Onsets and Frames (2018) — linea base bibliografica | — | 94.80% (publicado) "
    "| 82.60% (publicado) | — | — |")
tabla = chr(10).join(lines)
print(tabla)
Path("/kaggle/working/benchmark_full_tabla.md").write_text(tabla)


## 9. Notas de trazabilidad

* La selección de obras es determinista (semilla 22779); la lista exacta queda en
  `benchmark_full_por_obra.csv` (columnas `composer`, `title`, `work`).
* Este benchmark sustituye al preliminar de `notebooks/amt-benchmark.ipynb`
  (1 fragmento de 30 s), que se conserva como registro de la fase exploratoria.
* Al terminar la ejecución en Kaggle, descargar los CSV y la tabla Markdown y
  actualizar `docs/literature_review_amt.md` §§6–8 con: número de obras, tabla
  agregada y decisión final verificada contra los umbrales del hito H2.